# Tutorial 6: Migrating from Legacy Format

This tutorial covers how to migrate data and models from the legacy deepSCENIC format (pre package release) to the new unified format. It's only relevant if you used to work in the legacy codebase directly. 

## Setup

In [1]:
import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

deepSCENIC version: 0.1.0


## Legacy vs New Format

### Legacy Format (Multiple Files)

The legacy format stored models across multiple separate files:

**Model files:**
- `best_model.pth` - VAE checkpoint (encoder, decoders, E2 weights)
- `best_model_tf2r.pth` - MotifNet checkpoint
- `best_model_tf2r_encoder.pth` - Enformer checkpoint (~1GB)
- `E1.pkl` - pre-computed TF→region matrix
- `E2.pkl` - pre-computed region→gene matrix

### New Format (Unified)

**Model:** Single `.pt` file containing:
- All model weights (VAE, MotifNet, Enformer)
- Pre-computed E1 matrix
- Feature names (genes, regions, TFs)
- Training configuration

**Data:** A single .h5mu file. We don't have any specific data conversion functions, but expect the user to run the preprocessing tutorial on their raw data to bring that data up-to-code.   

## Step 1: Loading Legacy Data

Load the preprocessed data from the first tutorial. There is no `load_legacy_data` function, we expect the users to perform these steps themselves.  

In [2]:
# Set your legacy data directory
import os

data_dir = "../../../../data/deepSCENIC/MM_lines/"
assert os.path.exists(data_dir), "dir not found"

mdata = ds.read(data_dir + "preprocessed_data.h5mu")

deepscenic.io - INFO - Loaded ../../../../data/deepSCENIC/MM_lines/preprocessed_data.h5mu: 936 cells, 10595 genes, 281820 regions


## Step 2: Loading Legacy Models

Use `ds.tl.load_legacy_model()` to load a trained model from the legacy 3-file format.

**Requirements:**
- The 3 model checkpoint files (.pth)
- Preprocessed MuData with `is_tf` and `r2g` (see Tutorial 01)
- The pre-computed E1 matrix (E1.pkl) (optional)  
If the E1 matrix is not provided, we will attempt to recalculate this (you'll have to register a genome first).

In [ ]:
# Set your paths
legacy_results_dir = "../../../../data/deepSCENIC/MM_lines_legacy_results/MM_lines"

# Load legacy model
model = ds.tl.load_legacy_model(
    # Model checkpoints
    vae_path=f"{legacy_results_dir}/best_model.pth",
    tf2rnet_path=f"{legacy_results_dir}/best_model_tf2r.pth",
    enformer_path=f"{legacy_results_dir}/best_model_tf2r_encoder.pth",
    # Preprocessed data (contains TF annotations, r2g matrix)
    mdata=mdata,
    # Pre-computed E1 matrix (optional, leave None to recalculate)
    e1_path=f"{legacy_results_dir}/E1.pkl",
    # Device
    device="cuda",  # or "cpu"
)

print(f"Model loaded!")
print(f"  TFs: {len(model.tf_names)}")
print(f"  Genes: {len(model.gene_names)}")
print(f"  Regions: {len(model.region_names)}")
print(f"  E1 shape: {model.adj_E1.shape}")

In [ ]:
# Alternative: Reconstruct E1 from model weights (if E1.pkl is missing)
# This requires a registered genome and takes longer
#
# ds.register_genome("/path/to/hg38.fa")
# model = ds.tl.load_legacy_model(
#     vae_path=f"{legacy_results_dir}/best_model.pth",
#     tf2rnet_path=f"{legacy_results_dir}/best_model_tf2r.pth",
#     enformer_path=f"{legacy_results_dir}/best_model_tf2r_encoder.pth",
#     mdata=mdata,
#     e1_path=None,  # Will reconstruct from Enformer + MotifNet
#     device="cuda",
# )

### Loading with PPI Network

If the legacy model was trained with PPI (protein-protein interaction) data, ensure your MuData has PPI indices computed via `ds.pp.build_ppi_network()`. The function will automatically detect and use PPI if present in `mdata.uns`.

In [ ]:
# First, build PPI network in MuData (if not already done)
# import pandas as pd
# ppi_data = pd.read_csv("path/to/ppi_interactions.csv")
# ds.pp.build_ppi_network(mdata, ppi_data, confidence=0.5, species="human")

# Then load legacy model - PPI will be automatically detected
# model_with_ppi = ds.tl.load_legacy_model(
#     vae_path=f"{results_dir}/best_model.pth",
#     tf2rnet_path=f"{results_dir}/best_model_tf2r.pth",
#     enformer_path=f"{results_dir}/best_model_tf2r_encoder.pth",
#     mdata=mdata,  # Now contains PPI indices
#     e1_path=f"{results_dir}/E1.pkl",
# )
#
# print(f"PPI enabled: {model_with_ppi.config.use_ppi}")

## Step 3: Loading GRN Matrices Only

If you only need the GRN matrices for analysis (without loading the full model), use `ds.tl.load_legacy_grn()`:

In [ ]:
# Load just the GRN matrices
# grn = ds.tl.load_legacy_grn(
#     e1_path=f"{results_dir}/E1.pkl",
#     e2_path=f"{results_dir}/E2.pkl",
# )
#
# # E1: TF -> region binding matrix
# print(f"E1 shape: {grn['E1'].shape}")  # (n_tfs, n_regions)
# print(f"E1 columns (regions): {grn['E1'].columns[:5].tolist()}")
# print(f"E1 index (TFs): {grn['E1'].index[:5].tolist()}")
#
# # E2: region -> gene regulatory matrix
# print(f"E2 shape: {grn['E2'].shape}")  # (n_regions, n_genes)
# print(f"E2 columns (genes): {grn['E2'].columns[:5].tolist()}")

This is useful for:
- Analyzing TF-target relationships without GPU
- Visualizing GRN networks
- Computing TF importance scores
- Comparing GRNs between conditions

## Step 4: Saving in New Format

Once loaded, save your data and model in the new unified format for easier future use.

### Saving Data

In [ ]:
# Save MuData to new format
# mdata.write("converted_data.h5mu")
#
# # Verify by reloading
# mdata_reloaded = ds.read("converted_data.h5mu")
# print(f"Reloaded: {mdata_reloaded.n_obs} cells")

### Saving Model

In [ ]:
# Save model to new unified format
# model.save("converted_model.pt")
#
# # Verify by reloading
# model_reloaded = ds.tl.load_model("converted_model.pt")
# print(f"Reloaded model with {len(model_reloaded.tf_names)} TFs")

```{note}
The new model format (~1GB) includes all components in a single file:
- VAE, MotifNet, and Enformer weights
- Pre-computed E1 matrix
- Feature names and training config

Future loading only requires: `model = ds.tl.load_model("model.pt")`
```

## Summary

| Task | Legacy | New |
|------|--------|-----|
| Load data | Multiple files | `ds.read("data.h5mu")` |
| Load model | 3 checkpoints + data files | `ds.tl.load_model("model.pt")` |
| Convert model | `ds.tl.load_legacy_model(checkpoints, mdata, e1_path=...)` | N/A |

The key changes:
- `load_legacy_model()` now takes a preprocessed MuData object instead of separate data file paths
- `e1_path` is now optional - if not provided, E1 will be reconstructed from the model weights (requires `ds.register_genome()` first)

## Next Steps

After converting your data and model:

- {doc}`03_grn_analysis` - Analyze the gene regulatory network
- {doc}`04_perturbation_simulation` - Simulate TF perturbations